# 02 — BitNet Architecture Lab

Prototype and explore the **BitNet** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.bitnet.config import BitNetConfig
from src.models.bitnet.model import BitNetSLM
from src.core.generation import generate
from src.utils.training import count_params

In [ ]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = BitNetConfig(
    vocab_size=50257, block_size=32,
    n_layer=2, n_head=2, n_kv_head=1, n_embd=64,
    intermediate_size=128, dropout=0.0,
)
model = BitNetSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

In [ ]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

In [ ]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

In [ ]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'bitnet_tiny':   BitNetConfig(vocab_size=50257, block_size=16,  n_layer=2,  n_head=2,  n_kv_head=1, n_embd=64,  intermediate_size=128),
    'bitnet_small':  BitNetConfig(vocab_size=50257, block_size=128, n_layer=6,  n_head=6,  n_kv_head=2, n_embd=384, intermediate_size=1024),
    'bitnet_medium': BitNetConfig(vocab_size=50257, block_size=256, n_layer=12, n_head=12, n_kv_head=4, n_embd=768, intermediate_size=2048),
}
for name, cfg in configs.items():
    n = count_params(BitNetSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')